# FMC Co-Retrieval from MODIS using Earth Engine and PyTorch MLP

**Description:** This notebook retrieves **Fuel Moisture Content (FMC)** by jointly inverting
Equivalent Water Thickness (EWT) and Dry Matter Content (DMC&Cm) from MODIS MCD43A4 reflectance
data, with LAI from MCD15A3H and land-cover type from MCD12Q1.

**Method overview:**
1. Build a daily MODIS reflectance + LAI stack with quality control.
2. Compute spectral water index (SWI) and normalised spectral angle index (NSAI).
3. Run a pre-trained 4-layer MLP (tanh activation) → EWT, DMC → $FMC = EWT / DMC \times 100$.
4. Use vegetation-type-specific model weights and normalisation parameters.

**Requirements:**
- Google Earth Engine Python API (`ee`) authenticated with a GEE project.
- PyTorch (`torch`) for loading model weights.
- Pre-trained `.pkl` model files placed in the working directory.

**Authors:** Xiangyan Hu, Weimin Ju,  Jing M. Chen, Xinjia Zhang, Xingwen Quan, Xiang Zhang, Xuguang Tang, Meihong Fang

**License:** CC BY 4.0

## 0. Configuration

Edit these values before running.

In [23]:
# ============================================================
#  User-configurable parameters — change these for your project
# ============================================================

GEE_PROJECT_ID = "ee-xxxxxxx"          # Your Google Earth Engine project ID
GEE_ASSET_FOLDER = f"projects/{GEE_PROJECT_ID}/assets"  # Asset export destination

# Date range for retrieval
DATA_START = "2024-02-01"
DATA_END   = "2024-02-21"

# MCD12Q1 land-cover year (usually the year of DATA_START)
LC_YEAR = 2022

# Model file directory (set to the folder containing your .pkl files)
MODEL_DIR = "."

# Export parameters (used in the final section)
EXPORT_START_DATE = "2024-02-13"
EXPORT_END_DATE   = "2024-02-15"
EXPORT_SCALE_M    = 500          # metres per pixel
EXPORT_MAX_PIXELS = 1e13

# Scale factor constants
REFLECTANCE_SCALE = 0.0001       # MCD43A4 reflectance scale factor
LAI_SCALE         = 0.1          # MCD15A3H LAI scale factor
FMC_SCALE         = 100          # Scale FMC to integer percentage (0–400)

## 1. Environment Setup & Imports

In [2]:
import os
import sys
from datetime import datetime, timedelta

import ee
import geemap
import numpy as np
import torch
import torch.nn as nn
from torch.serialization import add_safe_globals

# -----------------------------------------------------------------
# Clean up proxy environment variables that may interfere with
# Earth Engine authentication in some network environments (e.g. Colab).
# -----------------------------------------------------------------
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    os.environ.pop(var, None)


Proxy environment variables cleared.


### 1.1 Authenticate & Initialise Earth Engine

The authentication method depends on your environment:
- **Google Colab:** Uses `google.colab.auth` (interactive browser login).
- **Local / Vertex AI / other:** Uses `ee.Authenticate()` or existing credentials.

If you are NOT running in Colab, comment out the `auth.authenticate_user()` line
and make sure you have run `ee.Authenticate()` once beforehand.

In [3]:
# ----- Authenticate (Colab: interactive; local: reuse saved credentials) -----
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication completed.")
except ImportError:
    # Not running in Colab — assume ee.Authenticate() has been run previously
    print("Running outside Colab; using existing Earth Engine credentials.")

# ----- Initialise Earth Engine -----
ee.Initialize(project=GEE_PROJECT_ID)
print(f"Earth Engine initialised with project '{GEE_PROJECT_ID}'.")
print("Test:", ee.Number(1).getInfo())

Colab authentication completed.
Earth Engine initialised with project 'ee-ahu1293659-10'.
Test: 1


### 1.2 Register PyTorch Classes for Safe Deserialisation

Needed to load the pre-trained `.pkl` models with `torch.load()`.

In [4]:
# -----------------------------------------------------------------
# Define the MLP architecture (must match the saved model exactly)
# -----------------------------------------------------------------
class MLP(nn.Module):
    """4-layer MLP with tanh activation.

    Architecture:
        Linear(10 -> 64) -> tanh
        Linear(64 -> 128) -> tanh
        Linear(128 -> 64) -> tanh
        Linear(64 -> 2)   -> tanh   (outputs: EWT_norm, Cm_norm)
    """
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(10, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 2)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        x = self.fc4(x)
        return x


# Register safe globals so torch.load() can deserialise the model
add_safe_globals([MLP, nn.Linear, nn.ReLU, nn.Tanh, nn.Sequential, nn.Module])


Safe globals registered.


## 2. Data Loading — MODIS Collections

In [24]:
# -----------------------------------------------------------------
# Load MODIS image collections
#   MCD43A4  — NBAR surface reflectance (500 m daily)
#   MCD15A3H — LAI / FPAR (500 m, 4-day composite)
#   MCD12Q1  — Land-cover type (500 m annual)
# -----------------------------------------------------------------
mcd43a4  = ee.ImageCollection("MODIS/061/MCD43A4").filterDate(DATA_START, DATA_END)
mcd15a3h = ee.ImageCollection("MODIS/061/MCD15A3H").filterDate(DATA_START, DATA_END)
mcd12q1  = ee.Image(f"MODIS/061/MCD12Q1/{LC_YEAR}_01_01")


MCD43A4  images: 20
MCD15A3H images: 5


### 2.1 IGBP Land-Cover Reclassification

Map the 17-class IGBP legend to 4 broad vegetation types:

| Value | Vegetation Type | Original IGBP Classes |
|-------|-----------------|----------------------|
| 1     | Grass           | 8, 9, 10, 12, 14 |
| 2     | Broadleaf Forest| 2, 4, 5            |
| 3     | Conifer Forest  | 1, 3               |
| 4     | Shrub           | 6, 7               |
| 255   | Other (masked)  | all others          |

In [25]:
igbp = mcd12q1.select("LC_Type1")

# Reclassify IGBP → 4 broad vegetation types; 255 = non-vegetated / water
reclassified = igbp.expression(
    "(b('LC_Type1') == 8  || b('LC_Type1') == 9  ||"
    " b('LC_Type1') == 10 || b('LC_Type1') == 12 ||"
    " b('LC_Type1') == 14) ? 1"
    ": (b('LC_Type1') == 2 || b('LC_Type1') == 4 ||"
    " b('LC_Type1') == 5)  ? 2"
    ": (b('LC_Type1') == 1 || b('LC_Type1') == 3)  ? 3"
    ": (b('LC_Type1') == 6 || b('LC_Type1') == 7)  ? 4"
    ": 255"
).rename("reclassified")

# Mask out non-vegetated pixels (255)
reclassified = reclassified.updateMask(reclassified.neq(255))


Land-cover reclassification complete.


## 3. Image Preprocessing Functions

In [26]:
# -----------------------------------------------------------------
#  3.1  Attach closest-in-time LAI to each MCD43A4 image
# -----------------------------------------------------------------
def add_lai_to_mcd43a4(image):
    """For each MCD43A4 image, find the temporally closest MCD15A3H LAI image
    within a ±4 day window and attach its Lai, FparLai_QC, and FparExtra_QC bands.

    If no LAI observation is available within the ±4 day window, the search window
    is progressively expanded until a valid LAI image is found.

    Pixels with LAI == 0 are masked (as described in the methodology).
    """
    date = ee.Date(image.get("system:time_start"))

    # Search window: [-4, +5) days around the reflectance image date
    lai_images = mcd15a3h.filterDate(
        date.advance(-4, "day"), date.advance(5, "day")
    )

    # Find the LAI image closest in time
    def _compute_difference(lai_img):
        lai_date = ee.Date(lai_img.get("system:time_start"))
        diff = date.difference(lai_date, "day").abs()
        return lai_img.set("date_difference", diff)

    lai_images = lai_images.map(_compute_difference)
    min_diff = lai_images.aggregate_min("date_difference")
    closest = lai_images.filter(ee.Filter.eq("date_difference", min_diff))
    lai_image = ee.ImageCollection(closest).first()

    # Select bands and match projection
    lai_image = lai_image.select(
        ["Lai", "FparLai_QC", "FparExtra_QC"]
    ).setDefaultProjection(image.projection())

    # Mask out LAI == 0 (required per paper methodology)
    lai_image = lai_image.updateMask(lai_image.select("Lai").gt(0))

    return image.addBands(lai_image)

In [27]:
# -----------------------------------------------------------------
#  3.2  Scale reflectance and LAI to physical units
# -----------------------------------------------------------------
def process_image(image):
    """Scale MCD43A4 reflectance bands (×0.0001) and LAI (×0.1) to physical values.

    Returns an image with:
        Nadir_Reflectance_Band1–7   (scaled reflectance)
        Lai                          (scaled LAI)
        FparLai_QC, FparExtra_QC    (LAI quality flags)
        BRDF_Albedo_Band_Mandatory_Quality_Band1–7  (reflectance quality flags)
    """
    REFLECTANCE_BANDS = ["1", "2", "3", "4", "5", "6", "7"]
    QUALITY_BANDS = [
        f"BRDF_Albedo_Band_Mandatory_Quality_Band{i}" for i in range(1, 8)
    ]

    processed = ee.Image()

    # Scale reflectance bands
    for idx in REFLECTANCE_BANDS:
        band_name = f"Nadir_Reflectance_Band{idx}"
        scaled = image.select(band_name).multiply(REFLECTANCE_SCALE)
        processed = processed.addBands(scaled)

    # Scale LAI band
    lai_scaled = image.select("Lai").multiply(LAI_SCALE)
    processed = processed.addBands(lai_scaled)

    # Copy QC bands
    qc_bands = image.select(
        ["FparLai_QC", "FparExtra_QC"] + QUALITY_BANDS
    )
    processed = processed.addBands(qc_bands)

    return processed.copyProperties(image, ["system:time_start"])

In [28]:
# -----------------------------------------------------------------
#  3.3  Quality-control masking and quality-level band
# -----------------------------------------------------------------
def QCprocess_image(image):
    """Apply MODIS quality flags to build a 4-level quality band.

    Quality levels (priority: 0 > 1 > 2 > 3):
        0 — Both reflectance AND LAI pass QC (best quality).
        1 — Reflectance passes QC; LAI is not considered.
        2 — Both reflectance and LAI have valid pixels (no QC requirement).
        3 — Only reflectance has valid pixels (lowest quality).
    """
    REFLECTANCE_BANDS = ["1", "2", "3", "4", "5", "6", "7"]

    # ---- Build a mask: all 7 reflectance bands have valid data ----
    reflectance_masks = []
    for idx in REFLECTANCE_BANDS:
        band_mask = image.select(f"Nadir_Reflectance_Band{idx}").mask()
        band_mask = band_mask.updateMask(band_mask)
        reflectance_masks.append(band_mask)

    reflectance_mask = reflectance_masks[0]
    for m in reflectance_masks[1:]:
        reflectance_mask = reflectance_mask.And(m)

    # ---- LAI valid-data mask ----
    lai_mask = image.select("Lai").mask()
    lai_mask = lai_mask.updateMask(lai_mask)

    # ---- MCD43A4 quality mask (bit 0 == 0 → best quality) ----
    quality_mask_MCD43A4 = image.select(
        "BRDF_Albedo_Band_Mandatory_Quality_Band1"
    ).bitwiseAnd(1).eq(0)
    for i in range(2, 8):
        band_qc = f"BRDF_Albedo_Band_Mandatory_Quality_Band{i}"
        current = image.select(band_qc).bitwiseAnd(1).eq(0)
        quality_mask_MCD43A4 = quality_mask_MCD43A4.And(current)
    quality_mask_MCD43A4 = quality_mask_MCD43A4.updateMask(quality_mask_MCD43A4)

    # ---- LAI quality mask (MCD15A3H QC bits) ----
    fpar_lai_qc = image.select("FparLai_QC")
    modland_qc_mask = fpar_lai_qc.bitwiseAnd(1).eq(0)              # Bit 0
    cloud_state_mask = fpar_lai_qc.rightShift(3).bitwiseAnd(3).eq(0)  # Bits 3-4

    fpar_extra_qc = image.select("FparExtra_QC")
    snow_ice_mask = fpar_extra_qc.rightShift(2).bitwiseAnd(1).eq(0)   # Bit 2
    cloud_shadow_mask = fpar_extra_qc.rightShift(6).bitwiseAnd(1).eq(0)  # Bit 6

    quality_mask_LAI = (
        modland_qc_mask.And(cloud_state_mask)
        .And(snow_ice_mask)
        .And(cloud_shadow_mask)
    )
    quality_mask_LAI = quality_mask_LAI.updateMask(quality_mask_LAI)

    # ---- Build hierarchical quality levels ----
    quality_level_0 = quality_mask_MCD43A4.And(quality_mask_LAI)   # best
    quality_level_1 = quality_mask_MCD43A4                          # reflectance QC only
    quality_level_2 = reflectance_mask.And(lai_mask)                # both have values
    quality_level_3 = reflectance_mask                              # reflectance only

    # Stack with priority (0 wins over 1 over 2 over 3)
    quality_band = (
        quality_level_3.where(quality_level_3, 3)
        .where(quality_level_2, 2)
        .where(quality_level_1, 1)
        .where(quality_level_0, 0)
    )

    return image.addBands(quality_band.rename("Quality_Level"))

## 4. Spectral Indices & Model Inference

In [29]:
# -----------------------------------------------------------------
#  4.1  Spectral Water Index (SWI)
# -----------------------------------------------------------------
def SACEWT(image):
    """Compute the Spectral Absorption Cosine of Equivalent Water Thickness (SWI)
    from bands 2 (859 nm) and 5 (1240 nm).

    Reference: Fang et al. (2017) — SWI = cos(θ) between the reflectance vector
    and the water absorption coefficient vector at 859 & 1240 nm.
    """
    # Central wavelengths (nm) and water absorption coefficients (cm⁻¹)
    all_bands = np.array([
        412, 443, 469, 488, 531, 551, 555, 645, 667, 678,
        748, 859, 869, 905, 936, 940, 1240, 1375, 1640, 2130
    ])
    all_absorbing = np.array([
        0.000202945, 0.000208502, 0.000131038, 0.00022918,
        0.000497591, 0.000594837, 0.00062043,  0.003341514,
        0.005383188, 0.005862709, 0.026663944, 0.043961961,
        0.047051415, 0.069649995, 0.157152966, 0.197166888,
        1.146759147, 9.053919179, 6.154778038, 25.68018038
    ])

    # Select absorption coefficients at 859 nm and 1240 nm
    midband = [859, 1240]
    indices = [np.where(all_bands == v)[0][0] for v in midband]
    a1, a2 = all_absorbing[indices]

    band2 = image.select("Nadir_Reflectance_Band2")   # 859 nm
    band5 = image.select("Nadir_Reflectance_Band5")   # 1240 nm

    # SWI = (r₂·a₂ + r₅·a₅) / (||r|| · ||a||)
    result = image.expression(
        "numerator / denominator",
        {
            "numerator": image.expression(
                "r2 * a1 + r5 * a2",
                {"r2": band2, "r5": band5, "a1": a1, "a2": a2}
            ),
            "denominator": image.expression(
                "sqrt(r2 * r2 + r5 * r5) * sqrt(a1 * a1 + a2 * a2)",
                {"r2": band2, "r5": band5, "a1": a1, "a2": a2}
            ),
        },
    ).rename("SWI")

    return image.addBands(result)

In [30]:
# -----------------------------------------------------------------
#  4.2  Normalised Spectral Angle Index (NSAI)
# -----------------------------------------------------------------
def AI_vector(image, scale):
    """Compute the Normalised Spectral Angle Index (NSAI) using a
    vegetation-type-specific scale factor.
    Reference: Fang et al. (2023).

    Uses bands 1 (645 nm), 2 (859 nm), 5 (2130 nm).
    NSAI = acos(mm / dd) / π,  the normalised angle between the
    reflectance vector and the scaled reference vector.
    """
    R1, R2, R3 = 645.0, 859.0, 2130.0   # reference wavelengths (nm)
    r1_s, r2_s, r3_s = R1 * scale, R2 * scale, R3 * scale

    a1 = image.select("Nadir_Reflectance_Band1")   # ~645 nm
    a2 = image.select("Nadir_Reflectance_Band2")   # ~859 nm
    a3 = image.select("Nadir_Reflectance_Band5")   # ~2130 nm

    nsai = image.expression(
        "acos(mm / dd) / 3.141592653589793",
        {
            "dd": image.expression(
                "ee * cc",
                {
                    "ee": image.expression(
                        "sqrt((r3_s - r2_s) ** 2 + (a3 - a2) ** 2)",
                        {"r3_s": r3_s, "r2_s": r2_s, "a3": a3, "a2": a2},
                    ),
                    "cc": image.expression(
                        "sqrt((r1_s - r2_s) ** 2 + (a1 - a2) ** 2)",
                        {"r1_s": r1_s, "r2_s": r2_s, "a1": a1, "a2": a2},
                    ),
                },
            ),
            "mm": image.expression(
                "(r1_s - r2_s) * (r3_s - r2_s) + (a1 - a2) * (a3 - a2)",
                {
                    "r1_s": r1_s, "r2_s": r2_s, "r3_s": r3_s,
                    "a1": a1, "a2": a2, "a3": a3,
                },
            ),
        },
    ).rename("NSAI")

    return image.addBands(nsai)

In [31]:
# -----------------------------------------------------------------
#  4.3  Min–max normalisation to [−1, 1]
# -----------------------------------------------------------------
def normalize_band(image, band_name, x_min, x_max):
    """Normalise a single band to [−1, 1] using (x − min)/(max − min) × 2 − 1.

    The normalised band is added with the suffix '_Norm'.
    """
    band = image.select(band_name)
    x_min_img = ee.Image.constant(x_min)
    x_max_img = ee.Image.constant(x_max)
    norm = band.subtract(x_min_img).divide(x_max_img.subtract(x_min_img)) \
                .multiply(2).subtract(1)
    norm = norm.rename(f"{band_name}_Norm")
    return image.addBands(norm)

In [32]:
# -----------------------------------------------------------------
#  4.4  Model loading & weight extraction
# -----------------------------------------------------------------
def load_model(model_path):
    """Load a PyTorch model from a .pkl file (CPU)."""
    return torch.load(model_path, map_location=torch.device("cpu"))


def extract_weights(state_dict):
    """Extract fc1–fc4 weights and biases from the model state_dict as NumPy arrays."""
    keys = ["fc1.weight", "fc1.bias",
            "fc2.weight", "fc2.bias",
            "fc3.weight", "fc3.bias",
            "fc4.weight", "fc4.bias"]
    return tuple(state_dict[k].numpy() for k in keys)


def apply_model(image, feature_band_names, weights):
    """Run the MLP forward pass inside Earth Engine using array operations.

    Parameters
    ----------
    image : ee.Image
        Image with normalised input bands.
    feature_band_names : list[str]
        Ordered list of input band names (must match model training order).
    weights : tuple[ndarray × 8]
        (fc1_w, fc1_b, fc2_w, fc2_b, fc3_w, fc3_b, fc4_w, fc4_b).

    Returns
    -------
    ee.Image with bands 'output1' (EWT_norm) and 'output2' (Cm_norm).
    """
    fc1_w, fc1_b, fc2_w, fc2_b, fc3_w, fc3_b, fc4_w, fc4_b = weights

    # Convert NumPy weights to Earth Engine array images
    fc1_w_ee = ee.Image(ee.Array(fc1_w.tolist()))
    fc1_b_ee = ee.Image(ee.Array(fc1_b.tolist())).toArray(1)
    fc2_w_ee = ee.Image(ee.Array(fc2_w.tolist()))
    fc2_b_ee = ee.Image(ee.Array(fc2_b.tolist())).toArray(1)
    fc3_w_ee = ee.Image(ee.Array(fc3_w.tolist()))
    fc3_b_ee = ee.Image(ee.Array(fc3_b.tolist())).toArray(1)
    fc4_w_ee = ee.Image(ee.Array(fc4_w.tolist()))
    fc4_b_ee = ee.Image(ee.Array(fc4_b.tolist())).toArray(1)

    # Stack input bands into an array image of shape [n_features, 1]
    img_array = image.select(feature_band_names).toArray().toArray(1)

    # tanh activation expressed in Earth Engine primitives
    def _tanh(x):
        exp_x = x.exp()
        exp_neg_x = x.multiply(-1).exp()
        return exp_x.subtract(exp_neg_x).divide(exp_x.add(exp_neg_x))

    # Layer 1: fc1 → tanh
    t = fc1_w_ee.matrixMultiply(img_array).add(fc1_b_ee)
    t = _tanh(t)
    # Layer 2: fc2 → tanh
    t = fc2_w_ee.matrixMultiply(t).add(fc2_b_ee)
    t = _tanh(t)
    # Layer 3: fc3 → tanh
    t = fc3_w_ee.matrixMultiply(t).add(fc3_b_ee)
    t = _tanh(t)
    # Layer 4: fc4 (output, no activation applied here; tanh is in the model)
    t = fc4_w_ee.matrixMultiply(t).add(fc4_b_ee)

    # Extract the 2 output neurons
    t1 = t.arrayGet([0, 0])
    t2 = t.arrayGet([1, 0])

    return ee.Image.cat([t1, t2]).rename(["output1", "output2"])

### 4.5 Core Retrieval Pipeline

The two functions below are nearly identical except that
`process_vegetation_type_noLAI` omits the LAI band from the input features
(for pixels where LAI quality is insufficient).

In [33]:
# ---- Shared band lists ----
BANDS_WITH_LAI = [
    "Nadir_Reflectance_Band3", "Nadir_Reflectance_Band4",
    "Nadir_Reflectance_Band1", "Nadir_Reflectance_Band2",
    "Nadir_Reflectance_Band5", "Nadir_Reflectance_Band6",
    "Nadir_Reflectance_Band7", "Lai", "NSAI", "SWI",
]
BANDS_WITHOUT_LAI = [
    "Nadir_Reflectance_Band3", "Nadir_Reflectance_Band4",
    "Nadir_Reflectance_Band1", "Nadir_Reflectance_Band2",
    "Nadir_Reflectance_Band5", "Nadir_Reflectance_Band6",
    "Nadir_Reflectance_Band7", "NSAI", "SWI",
]


def _retrieval_pipeline(image, x_min, x_max, model_path, nsai_scale,
                        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list):
    """Shared FMC retrieval pipeline for one vegetation type.

    Steps:
        1. Compute SWI and NSAI.
        2. Normalise all input bands to [−1, 1].
        3. Run the MLP → EWT_norm, Cm_norm.
        4. Inverse-normalise to physical EWT, Cm.
        5. Compute FMC = EWT / Cm × 100, clamped to [0, 400] as integer.
    """
    model = load_model(model_path)
    weights = extract_weights(model.state_dict())

    # Spectral indices
    image = SACEWT(image)
    image = AI_vector(image, nsai_scale)

    # Normalise each input band
    for band_name in band_list:
        image = normalize_band(image, band_name, x_min[band_name], x_max[band_name])

    # MLP inference
    feature_bands = [f"{b}_Norm" for b in band_list]
    image = image.addBands(
        apply_model(image, feature_bands, weights).rename("EWT_Norm", "Cm_Norm")
    )

    # Inverse normalisation:  y = (y_norm + 1) / 2 × (ymax − ymin) + ymin
    denorm_expr = "max(min((a + 1) * (b - c) / 2 + c, b), c)"
    image = image.addBands(
        image.expression(denorm_expr, {
            "a": image.select("EWT_Norm"),
            "b": ewt_ymax, "c": ewt_ymin,
        }).rename("EWT")
    )
    image = image.addBands(
        image.expression(denorm_expr, {
            "a": image.select("Cm_Norm"),
            "b": cm_ymax, "c": cm_ymin,
        }).rename("Cm")
    )

    # FMC = EWT / Cm × 100, clamped to [0, 4] then scaled to integer [0, 400]
    fmc_expr = "int(max(min(a / b, 4), 0) * 100)"
    image = image.addBands(
        image.expression(fmc_expr, {
            "a": image.select("EWT"),
            "b": image.select("Cm"),
        }).rename("FMC")
    )

    return image.select("FMC", "reclassified", "EWT", "Cm")


def process_vegetation_type(image, x_min, x_max, model_path, nsai_scale,
                            ewt_ymax, ewt_ymin, cm_ymax, cm_ymin):
    """FMC retrieval WITH LAI as an input feature."""
    return _retrieval_pipeline(
        image, x_min, x_max, model_path, nsai_scale,
        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, BANDS_WITH_LAI,
    )


def process_vegetation_type_noLAI(image, x_min, x_max, model_path, nsai_scale,
                                  ewt_ymax, ewt_ymin, cm_ymax, cm_ymin):
    """FMC retrieval WITHOUT LAI (for pixels where LAI QC is poor)."""
    return _retrieval_pipeline(
        image, x_min, x_max, model_path, nsai_scale,
        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, BANDS_WITHOUT_LAI,
    )

### 4.6 Vegetation-Type Parameters

Each vegetation type has its own:
- NSAI scale factor (calibrated per type)
- EWT / Cm min–max for inverse normalisation
- Per-band Xmin / Xmax for input normalisation
- Paths to trained model weights (with-LAI and without-LAI variants)

In [34]:
import os as _os

VEGETATION_PARAMS = {
    # ── Type 2: Broadleaf Forest ──
    "Broadleaf": {
        "type_num": 2,
        "NSAI_scale": 0.0005131208685522245,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 1.62939124e-08,
            "Nadir_Reflectance_Band2": 4.62736392e-02,
            "Nadir_Reflectance_Band3": 1.66886764e-09,
            "Nadir_Reflectance_Band4": 6.31038871e-06,
            "Nadir_Reflectance_Band5": 3.55455734e-02,
            "Nadir_Reflectance_Band6": 9.95534192e-03,
            "Nadir_Reflectance_Band7": 9.87126742e-09,
            "Lai":  1.00000000e-01,
            "NSAI": 4.14757000e-01,
            "SWI":  5.75963083e-01,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.3194022,
            "Nadir_Reflectance_Band2": 0.56783228,
            "Nadir_Reflectance_Band3": 0.12831889,
            "Nadir_Reflectance_Band4": 0.24003917,
            "Nadir_Reflectance_Band5": 0.81795011,
            "Nadir_Reflectance_Band6": 0.88806936,
            "Nadir_Reflectance_Band7": 0.7619845,
            "Lai":  8.1,
            "NSAI": 0.894215,
            "SWI":  0.85765764,
        },
        "model_path":         _os.path.join(MODEL_DIR, "Broadleafr-LAI.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Broadleaf-noLAI.pkl"),
    },
    # ── Type 4: Shrub ──
    "Shrub": {
        "type_num": 4,
        "NSAI_scale": 0.0005047165283853842,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 1.65210391e-02,
            "Nadir_Reflectance_Band2": 5.71611167e-02,
            "Nadir_Reflectance_Band3": 2.25194001e-08,
            "Nadir_Reflectance_Band4": 8.10305568e-03,
            "Nadir_Reflectance_Band5": 6.78616153e-02,
            "Nadir_Reflectance_Band6": 5.69148747e-02,
            "Nadir_Reflectance_Band7": 4.76390793e-02,
            "Lai":  1.00000000e-01,
            "NSAI": 5.97562000e-01,
            "SWI":  7.14518778e-01,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.30873206,
            "Nadir_Reflectance_Band2": 0.56178756,
            "Nadir_Reflectance_Band3": 0.12600185,
            "Nadir_Reflectance_Band4": 0.23361443,
            "Nadir_Reflectance_Band5": 0.81422779,
            "Nadir_Reflectance_Band6": 0.89383039,
            "Nadir_Reflectance_Band7": 0.76591537,
            "Lai":  8.1,
            "NSAI": 0.894797,
            "SWI":  0.85769151,
        },
        "model_path":         _os.path.join(MODEL_DIR, "Shrub-LAI.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Shrub-noLAI.pkl"),
    },
    # ── Type 3: Conifer Forest ──
    "Conifer": {
        "type_num": 3,
        "NSAI_scale": 0.00045655039292557463,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 5.43104088e-04,
            "Nadir_Reflectance_Band2": 3.09274501e-02,
            "Nadir_Reflectance_Band3": 2.24034675e-09,
            "Nadir_Reflectance_Band4": 1.78789333e-03,
            "Nadir_Reflectance_Band5": 2.66043544e-02,
            "Nadir_Reflectance_Band6": 1.16602146e-02,
            "Nadir_Reflectance_Band7": 9.78448136e-06,
            "Lai":  1.00000000e-01,
            "NSAI": 5.48743000e-01,
            "SWI":  5.89962359e-01,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.30012971,
            "Nadir_Reflectance_Band2": 0.55365491,
            "Nadir_Reflectance_Band3": 0.12960828,
            "Nadir_Reflectance_Band4": 0.22935912,
            "Nadir_Reflectance_Band5": 0.78723129,
            "Nadir_Reflectance_Band6": 0.80551743,
            "Nadir_Reflectance_Band7": 0.67798712,
            "Lai":  8.1,
            "NSAI": 0.921687,
            "SWI":  0.85029598,
        },
        "model_path":         _os.path.join(MODEL_DIR, "Conifer-LAI.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Conifer-noLAI.pkl"),
    },
    # ── Type 1: Grass ──
    "Grass": {
        "type_num": 1,
        "NSAI_scale": 0.0003723398826046051,
        "EWT_Ymax": 0.097, "EWT_Ymin": 0.001,
        "Cm_Ymax":  0.053, "Cm_Ymin":  0.001,
        "Xmin": {
            "Nadir_Reflectance_Band1": 9.46770568e-08,
            "Nadir_Reflectance_Band2": 9.71318258e-02,
            "Nadir_Reflectance_Band3": 5.38758301e-09,
            "Nadir_Reflectance_Band4": 1.07576666e-03,
            "Nadir_Reflectance_Band5": 7.15844136e-02,
            "Nadir_Reflectance_Band6": 1.03212653e-02,
            "Nadir_Reflectance_Band7": 2.79864875e-08,
            "Lai":  1.00000000e-01,
            "NSAI": 2.93991000e-01,
            "SWI":  4.74718405e-01,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.23133421,
            "Nadir_Reflectance_Band2": 0.55292475,
            "Nadir_Reflectance_Band3": 0.10643674,
            "Nadir_Reflectance_Band4": 0.16479264,
            "Nadir_Reflectance_Band5": 0.570653,
            "Nadir_Reflectance_Band6": 0.53004359,
            "Nadir_Reflectance_Band7": 0.50089978,
            "Lai":  8.1,
            "NSAI": 0.786934,
            "SWI":  0.84677844,
        },
        "model_path":         _os.path.join(MODEL_DIR, "Grass-LAI.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Grass-noLAI.pkl"),
    },
}

# Lookup: vegetation type number → param dict key
VEG_TYPE_NUM_TO_KEY = {v["type_num"]: k for k, v in VEGETATION_PARAMS.items()}

### 4.7 Vegetation-Type Dispatch

These functions select the correct model parameters for a given vegetation type
and run the retrieval pipeline. The `noLAI` variant skips the LAI input feature.

In [35]:
def _run_for_type(image, type_num, use_lai):
    """Run the retrieval pipeline for a single vegetation type.

    Parameters
    ----------
    type_num : int  — 1=Grass, 2=Broadleaf, 3=Conifer, 4=Shrub
    use_lai  : bool — if True, include LAI in the input features
    """
    key = VEG_TYPE_NUM_TO_KEY[type_num]
    params = VEGETATION_PARAMS[key]
    model_path = params["model_path"] if use_lai else params["model_path_noLAI"]
    pipeline = process_vegetation_type if use_lai else process_vegetation_type_noLAI
    return pipeline(
        image=image,
        x_min=params["Xmin"],
        x_max=params["Xmax"],
        model_path=model_path,
        nsai_scale=params["NSAI_scale"],
        ewt_ymax=params["EWT_Ymax"],
        ewt_ymin=params["EWT_Ymin"],
        cm_ymax=params["Cm_Ymax"],
        cm_ymin=params["Cm_Ymin"],
    )


def vegetation_type(image, type_num):
    """Run FMC retrieval WITH LAI for a given vegetation type number."""
    return _run_for_type(image, type_num, use_lai=True)


def vegetation_type_noLAI(image, type_num):
    """Run FMC retrieval WITHOUT LAI for a given vegetation type number."""
    return _run_for_type(image, type_num, use_lai=False)

## 5. Per-Pixel Model Selection & Compositing

In [36]:
def add_reclassified(image):
    """Attach the global reclassified land-cover band to an image."""
    return image.addBands(reclassified.rename("reclassified"))


def get_vegetation_data(image):
    """For each pixel, select the appropriate vegetation-type model (with or
    without LAI based on quality level) and return FMC, EWT, Cm, Quality_Level.

    Quality-level → model selection:
        Level 0 or 2 → LAI is available → use with-LAI model
        Level 1 or 3 → LAI is unavailable/poor → use without-LAI model

    NOTE: This function computes all 8 model variants (4 veg types × 2 LAI
    variants) for every input image, then uses .where() to select per-pixel.
    This is computationally expensive. For large-scale production, consider
    splitting the image by vegetation type first (see optimisation note below).
    """
    quality_level = image.select("Quality_Level")
    reclassified_band = image.select("reclassified")

    # Masks for model selection
    has_lai_mask = quality_level.eq(0).Or(quality_level.eq(2))
    no_lai_mask  = quality_level.eq(1).Or(quality_level.eq(3))

    # Run all 8 model variants
    veg_results = {}
    for type_num in [1, 2, 3, 4]:
        with_lai = vegetation_type(image, type_num)
        without_lai = vegetation_type_noLAI(image, type_num)
        # Per-pixel: pick the with-LAI or without-LAI result
        combined = with_lai.where(no_lai_mask, without_lai)
        veg_results[type_num] = combined

    # Extract FMC, EWT, Cm for each vegetation type
    def _get_fmc_ewt_cm(veg_img):
        return (
            veg_img.select("FMC"),
            veg_img.select("EWT"),
            veg_img.select("Cm"),
        )

    fmc = {t: _get_fmc_ewt_cm(v)[0] for t, v in veg_results.items()}
    ewt = {t: _get_fmc_ewt_cm(v)[1] for t, v in veg_results.items()}
    cm  = {t: _get_fmc_ewt_cm(v)[2] for t, v in veg_results.items()}

    # Composite: select the correct vegetation-type result per pixel
    fmc_image = (
        fmc[1].where(reclassified_band.eq(1), fmc[1])
              .where(reclassified_band.eq(2), fmc[2])
              .where(reclassified_band.eq(3), fmc[3])
              .where(reclassified_band.eq(4), fmc[4])
              .rename("FMC")
    )
    ewt_image = (
        ewt[1].where(reclassified_band.eq(1), ewt[1])
              .where(reclassified_band.eq(2), ewt[2])
              .where(reclassified_band.eq(3), ewt[3])
              .where(reclassified_band.eq(4), ewt[4])
              .rename("EWT")
    )
    cm_image = (
        cm[1].where(reclassified_band.eq(1), cm[1])
              .where(reclassified_band.eq(2), cm[2])
              .where(reclassified_band.eq(3), cm[3])
              .where(reclassified_band.eq(4), cm[4])
              .rename("Cm")
    )

    # Mask non-vegetated pixels (reclassified == 255)
    veg_mask = reclassified_band.neq(255)
    fmc_image = fmc_image.updateMask(veg_mask)
    ewt_image = ewt_image.updateMask(veg_mask)
    cm_image  = cm_image.updateMask(veg_mask)
    quality_level = quality_level.updateMask(veg_mask)

    return (
        fmc_image
        .addBands([ewt_image, cm_image])
        .addBands(quality_level.rename("Quality_Level"))
        .updateMask(veg_mask)
    )

## 6. Run the Retrieval Pipeline

In [37]:
# -----------------------------------------------------------------
# Chain all processing steps via .map() over the image collection
# -----------------------------------------------------------------
print("Running retrieval pipeline...")

mcd43a4_add_lai      = mcd43a4.map(add_lai_to_mcd43a4)
mcd43a4_scaled       = mcd43a4_add_lai.map(process_image)
mcd43a4_qc           = mcd43a4_scaled.map(QCprocess_image)
mcd43a4_with_reclass = mcd43a4_qc.map(add_reclassified)
fmc_result           = mcd43a4_with_reclass.map(get_vegetation_data)

# ---- Inspect the first result ----
first_image = fmc_result.first()
band_names = first_image.bandNames().getInfo()
print(f"Output bands: {band_names}")

Running retrieval pipeline...
Output bands: ['FMC', 'EWT', 'Cm', 'Quality_Level']


## 7. Visualise (Optional)

Use `geemap` to quickly inspect the FMC map for one day.

In [ ]:
import geemap
# ---------------------------------
# Define region of interest (ROI)
# ---------------------------------
roi = ee.Geometry.Rectangle([-118.8, 33.5, -117.5, 34.5])  # Southern California
first_clip = first_image.clip(roi)

# ---------------------------------
# Visualization parameters
# ---------------------------------
fmc_vis = {
    "bands": ["FMC"],
    "min": 0,
    "max": 400,
    "palette": ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"],
}

# ---------------------------------
# Create map
# ---------------------------------
Map = geemap.Map()
Map.centerObject(roi, 8)
Map.addLayer(first_clip, fmc_vis, "FMC (ROI)")
Map.add_colorbar(fmc_vis, label="FMC (%)")

Map

## 8. Export to Earth Engine Assets

In [ ]:
# -----------------------------------------------------------------
# Export FMC + Quality_Level for each day in the configured range
# -----------------------------------------------------------------
GLOBAL_RECT = ee.Geometry.Rectangle([-180, -90, 180, 90], "EPSG:4326", False)

start_dt = datetime.strptime(EXPORT_START_DATE, "%Y-%m-%d")
end_dt   = datetime.strptime(EXPORT_END_DATE,   "%Y-%m-%d")

current_dt = start_dt
while current_dt < end_dt:
    date_str = current_dt.strftime("%Y-%m-%d")
    ee_start = ee.Date(date_str)
    ee_end   = ee_start.advance(1, "day")

    # Select FMC + Quality_Level for the day
    daily_image = (
        fmc_result
        .filterDate(ee_start, ee_end)
        .first()
        .select(["FMC", "Quality_Level"])
        .clip(GLOBAL_RECT)
    )

    asset_id = f"{GEE_ASSET_FOLDER}/{date_str}_FMC"

    task = ee.batch.Export.image.toAsset(
        image=daily_image,
        description=date_str,
        assetId=asset_id,
        crs="EPSG:4326",
        scale=EXPORT_SCALE_M,
        region=GLOBAL_RECT,
        maxPixels=EXPORT_MAX_PIXELS,
    )
    task.start()
    print(f"Export task for {date_str} started → {asset_id}")

    current_dt += timedelta(days=1)

print("All export tasks submitted.")